In [1]:
import os
from pyspark.sql import functions as f, SparkSession, types as t
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql.functions import udf, col, length, isnan, when, count, regexp_replace
from datetime import datetime

db_user = 'DB_202613_c_cantini'
db_psswd = '201923481'
source_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/WWImportersTransactional'
dest_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/DB_202613_c_cantini?rewriteBatchedStatements=true&useServerPrepStmts=false'

path_jar_driver = './drivers/mysql-connector-j.jar'

conf = SparkConf() \
    .set('spark.driver.extraClassPath', path_jar_driver) \
    .set('spark.jars', path_jar_driver)

spark_context = SparkContext(conf=conf)
sql_context = SQLContext(spark_context)
spark = sql_context.sparkSession

Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport


26/07/01 17:42:24 WARN Utils: Your hostname, codespaces-d5a85f resolves to a loopback address: 127.0.0.1; using 10.0.13.98 instead (on interface eth0)
26/07/01 17:42:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/07/01 17:42:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 17:42:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
/home/vscode/.local/lib/python3.11/site-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [2]:
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc')\
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd

def guardar_db(db_connection_string, df, tabla, db_user, db_psswd):
    df.select('*').write.format('jdbc') \
      .mode('append') \
      .option('url', db_connection_string) \
      .option('dbtable', tabla) \
      .option('user', db_user) \
      .option('password', db_psswd) \
      .option('driver', 'com.mysql.cj.jdbc.Driver') \
      .option('batchsize', 5000) \
      .save()

### Dimensión Producto

Esta dimensión es compartida con el hecho de Orden, por lo que se reutiliza sin cambios adicionales de negocio.

#### Extracción
Se extraen los datos de las tablas `Producto` y `Colores` de la base transaccional.

In [3]:
sql_productos = '''(SELECT DISTINCT ID_Producto as ID_Producto_T, ID_Color, NombreProducto, Marca, Necesita_refrigeracion, Dias_tiempo_entrega, Impuesto, PrecioUnitario, PrecioRecomendado FROM WWImportersTransactional.Producto) AS Temp_productos'''
sql_colores = '''(SELECT DISTINCT ID_Color, Color FROM WWImportersTransactional.Colores) AS Temp_colores'''

productos = obtener_dataframe_de_bd(source_db_connection_string, sql_productos, db_user, db_psswd)
colores = obtener_dataframe_de_bd(source_db_connection_string, sql_colores, db_user, db_psswd)

#### Transformación
Se hace join con la tabla de colores, se completan los productos sin color con el valor "Missing" y se genera la llave `ID_Producto_DWH`.

In [4]:
productos = productos.join(colores, how='left', on='ID_Color').fillna({'Color': 'Missing'})
productos = productos.coalesce(1).withColumn('ID_Producto_DWH', f.monotonically_increasing_id() + 1)
productos = productos.select('ID_Producto_DWH','ID_Producto_T','NombreProducto','Marca','Color',
                              'Necesita_refrigeracion','Dias_tiempo_entrega','PrecioRecomendado',
                              'Impuesto','PrecioUnitario')
productos.show(5)

+---------------+-------------+--------------------+---------+-----+----------------------+-------------------+-----------------+--------+--------------+
|ID_Producto_DWH|ID_Producto_T|      NombreProducto|    Marca|Color|Necesita_refrigeracion|Dias_tiempo_entrega|PrecioRecomendado|Impuesto|PrecioUnitario|
+---------------+-------------+--------------------+---------+-----+----------------------+-------------------+-----------------+--------+--------------+
|              1|           59|RC toy sedan car ...|Northwind|  Red|                     0|                 14|               37|      15|            25|
|              2|           64|RC vintage Americ...|Northwind|  Red|                     0|                 14|               45|      15|            30|
|              3|           68|Ride on toy sedan...|Northwind|  Red|                     0|                 14|              344|      15|           230|
|              4|           73|Ride on vintage A...|Northwind|  Red|        

#### Carga
Se guarda el resultado en la tabla `Producto` de la base de datos destino.

In [5]:
guardar_db(dest_db_connection_string, productos, 'Producto', db_user, db_psswd)
print("Producto guardado")

Producto guardado


### Dimensión Cliente

Esta dimensión también es compartida con el hecho de Orden. Se reutiliza sin cambios adicionales de negocio.

#### Extracción
Se extraen los datos de `Clientes`, `CategoriasCliente` y `GruposCompra` de la base transaccional.

In [6]:
sql_categoriasCliente = '''(SELECT DISTINCT ID_Categoria, NombreCategoria FROM WWImportersTransactional.CategoriasCliente) AS Temp_categoriasclientes'''
sql_gruposCompra = '''(SELECT DISTINCT ID_GrupoCompra, NombreGrupoCompra FROM WWImportersTransactional.GruposCompra) AS Temp_gruposcompra'''
sql_clientes = '''(SELECT DISTINCT ID_Cliente as ID_Cliente_T, Nombre, ClienteFactura, ID_Categoria, ID_GrupoCompra, ID_CiudadEntrega, LimiteCredito, FechaAperturaCuenta, DiasPago FROM WWImportersTransactional.Clientes) AS Temp_clientes'''

categoriasCliente = obtener_dataframe_de_bd(source_db_connection_string, sql_categoriasCliente, db_user, db_psswd)
gruposCompra = obtener_dataframe_de_bd(source_db_connection_string, sql_gruposCompra, db_user, db_psswd)
clientes = obtener_dataframe_de_bd(source_db_connection_string, sql_clientes, db_user, db_psswd)

#### Transformación
Se unen las categorías y grupos de compra, se genera la llave `ID_Cliente_DWH` y se agrega el registro comodín (ID=0) para casos sin coincidencia en el hecho.

In [7]:
clientes = clientes.join(gruposCompra, how='left', on='ID_GrupoCompra')
clientes = clientes.alias('cl').join(categoriasCliente.alias('ct'), how='left', on='ID_Categoria') \
    .select([col('cl.ID_Cliente_T'), col('cl.Nombre'), col('ct.NombreCategoria'), col('cl.NombreGrupoCompra'),
             col('cl.ClienteFactura'), col('cl.ID_CiudadEntrega'), col('cl.LimiteCredito'),
             col('cl.FechaAperturaCuenta'), col('cl.DiasPago')])

clientes = clientes.coalesce(1).withColumn('ID_Cliente_DWH', f.monotonically_increasing_id() + 1)
clientes = clientes.select('ID_Cliente_DWH','ID_Cliente_T','Nombre','NombreCategoria','NombreGrupoCompra',
                            'ClienteFactura','ID_CiudadEntrega','LimiteCredito','FechaAperturaCuenta','DiasPago')
clientes = clientes.fillna({'NombreCategoria':'Missing','NombreGrupoCompra':'Missing'})

# Registro comodín ID=0
clientes_0 = [('0','','Missing','Missing','Missing','0','0','','','')]
columns = ['ID_Cliente_DWH','ID_Cliente_T','Nombre','NombreCategoria','NombreGrupoCompra','ClienteFactura',
           'ID_CiudadEntrega','LimiteCredito','FechaAperturaCuenta','DiasPago']
cliente_0 = spark.createDataFrame(data=clientes_0, schema=columns)

clientes = clientes.union(cliente_0)
clientes = clientes.withColumn('ID_Cliente_DWH', col('ID_Cliente_DWH').cast('int')).orderBy(col('ID_Cliente_DWH'))
clientes.show(5)

+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|ID_Cliente_DWH|ID_Cliente_T|              Nombre|NombreCategoria|NombreGrupoCompra|ClienteFactura|ID_CiudadEntrega|LimiteCredito|FechaAperturaCuenta|DiasPago|
+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|             0|            |             Missing|        Missing|          Missing|             0|               0|             |                   |        |
|             1|         801|         Eric Torres|      Corporate|          Missing|           801|           31321|         3000|2013-01-01 00:00:00|       7|
|             2|         802|        Cosmina Vlad|      Corporate|          Missing|           802|            5192|         2940|2013-01-01 00:00:00|       7|
|             3|         803|          B

#### Carga
Se guarda el resultado en la tabla `Cliente` de la base de datos destino.

In [8]:
guardar_db(dest_db_connection_string, clientes, 'Cliente', db_user, db_psswd)
print("Cliente guardado")

Cliente guardado


### Dimensión Proveedor

Fuente: `proveedoresCopia` y `CategoriasProovedor`. Se aplican las reglas de negocio indicadas por WWI: se corrigen los valores negativos de `DiasPago` y se unifican los proveedores duplicados por error de digitación (mismo nombre con sufijo "Inc" o "Ltd").

#### Extracción
Se extraen los proveedores y sus categorías desde `WWImportersTransactional`.

In [9]:
sql_proveedores = '''(SELECT DISTINCT ProveedorID as ID_Proveedor_T, NombreProveedor, CategoriaProveedorID, DiasPago FROM WWImportersTransactional.proveedoresCopia) AS Temp_proveedores'''
sql_categoriasProveedor = '''(SELECT DISTINCT CategoriaProveedorID, CategoriaProveedor FROM WWImportersTransactional.CategoriasProveedores) AS Temp_catProveedor'''

proveedores = obtener_dataframe_de_bd(source_db_connection_string, sql_proveedores, db_user, db_psswd)
categoriasProveedor = obtener_dataframe_de_bd(source_db_connection_string, sql_categoriasProveedor, db_user, db_psswd)

In [10]:
sql_tablas2 = '''(SELECT TABLE_NAME FROM information_schema.TABLES 
                  WHERE TABLE_SCHEMA = 'WWImportersTransactional' 
                  AND (TABLE_NAME LIKE '%ransacc%' OR TABLE_NAME LIKE '%ovimient%')) AS Temp_tablas2'''
tablas2 = obtener_dataframe_de_bd(source_db_connection_string, sql_tablas2, db_user, db_psswd)
tablas2.show(20, truncate=False)

+----------------+
|TABLE_NAME      |
+----------------+
|TiposTransaccion|
|movimientos     |
|movimientosCopia|
+----------------+



#### Transformación
Se corrigen los días de pago negativos multiplicando por -1. Se limpia el nombre del proveedor quitando el sufijo "Inc" o "Ltd" para detectar los casos duplicados por error de digitación, y se genera la llave `ID_Proveedor_DWH` a partir del nombre ya limpio, de forma que los dos IDs originales del proveedor duplicado apunten al mismo registro en la dimensión.

In [11]:
from pyspark.sql import Window
from pyspark.sql.functions import dense_rank, regexp_replace, trim, first as f_first

# Corregir DiasPago negativos
proveedores = proveedores.withColumn('DiasPago', when(col('DiasPago') < 0, col('DiasPago') * -1).otherwise(col('DiasPago')))

# Limpiar nombre para unificar duplicados por digitación (mismo nombre + "Inc"/"Ltd")
proveedores = proveedores.withColumn('NombreProveedorLimpio',
                                      trim(regexp_replace(col('NombreProveedor'), '(?i)\\s+(Inc\\.?|Ltd\\.?)$', '')))

# Join con categoría
proveedores = proveedores.join(categoriasProveedor, how='left', on='CategoriaProveedorID')

# Llave DWH basada en el nombre limpio (mismo id para nombres unificados)
w = Window.orderBy('NombreProveedorLimpio')
proveedores = proveedores.withColumn('ID_Proveedor_DWH', dense_rank().over(w))

# Mapa ID transaccional -> ID DWH, se usa más adelante en el hecho
mapa_proveedor = proveedores.select('ID_Proveedor_T', 'ID_Proveedor_DWH')

# Dimensión final: un solo registro por proveedor ya unificado
proveedor_dim = proveedores.groupBy('ID_Proveedor_DWH').agg(
    f_first('NombreProveedorLimpio').alias('NombreProveedor'),
    f_first('CategoriaProveedor').alias('NombreCategoria'),
    f_first('DiasPago').alias('DiasPago')
)
proveedor_dim = proveedor_dim.fillna({'NombreCategoria': 'Missing'})

# Registro comodín ID=0
proveedor_0 = spark.createDataFrame([(0, 'Missing', 'Missing', 0)],
                                     schema=['ID_Proveedor_DWH','NombreProveedor','NombreCategoria','DiasPago'])
proveedor_dim = proveedor_dim.select('ID_Proveedor_DWH','NombreProveedor','NombreCategoria','DiasPago').union(proveedor_0)
proveedor_dim = proveedor_dim.withColumn('ID_Proveedor_DWH', col('ID_Proveedor_DWH').cast('int')).orderBy('ID_Proveedor_DWH')
proveedor_dim.show(5)

26/07/01 17:42:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:42:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:42:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:42:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:42:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:42:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 1

+----------------+--------------------+--------------------+--------+
|ID_Proveedor_DWH|     NombreProveedor|     NombreCategoria|DiasPago|
+----------------+--------------------+--------------------+--------+
|               0|             Missing|             Missing|       0|
|               1| A Datum Corporation| productos novedosos|      14|
|               2|Consolidated Mess...|servicios de mens...|      30|
|               3|             Contoso| productos novedosos|       7|
|               4|            Fabrikam|                ropa|      30|
+----------------+--------------------+--------------------+--------+
only showing top 5 rows



#### Carga
Se guarda el resultado en la tabla `Proveedor` de la base de datos destino.

In [12]:
guardar_db(dest_db_connection_string, proveedor_dim, 'Proveedor', db_user, db_psswd)
print("Proveedor guardado")

26/07/01 17:42:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:42:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:42:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:42:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:42:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:42:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 1

Proveedor guardado


### Dimensión TipoTransacción

Fuente: `TiposTransaccion`. Esta tabla ya fue validada previamente por el equipo de consultores de WWI, por lo que no requiere limpieza adicional.

#### Extracción
Se extraen los tipos de transacción desde `WWImportersTransactional`.

In [13]:
sql_tiposTransaccion = '''(SELECT DISTINCT TipoTransaccionID as ID_TipoTransaccion_T, TipoTransaccionNombre FROM WWImportersTransactional.TiposTransaccion) AS Temp_tiposTransaccion'''
tiposTransaccion = obtener_dataframe_de_bd(source_db_connection_string, sql_tiposTransaccion, db_user, db_psswd)

#### Transformación
Se genera la llave `ID_TipoTransaccion_DWH` y se agrega el registro comodín (ID=0).

In [14]:
tiposTransaccion = tiposTransaccion.coalesce(1).withColumn('ID_TipoTransaccion_DWH', f.monotonically_increasing_id() + 1)
tiposTransaccion = tiposTransaccion.select('ID_TipoTransaccion_DWH', 'ID_TipoTransaccion_T', 'TipoTransaccionNombre')

tipoTransaccion_0 = spark.createDataFrame([(0, 0, 'Missing')],
                                           schema=['ID_TipoTransaccion_DWH','ID_TipoTransaccion_T','TipoTransaccionNombre'])
tiposTransaccion = tiposTransaccion.union(tipoTransaccion_0)
tiposTransaccion = tiposTransaccion.withColumn('ID_TipoTransaccion_DWH', col('ID_TipoTransaccion_DWH').cast('int')).orderBy('ID_TipoTransaccion_DWH')
tiposTransaccion.show(20)

+----------------------+--------------------+---------------------+
|ID_TipoTransaccion_DWH|ID_TipoTransaccion_T|TipoTransaccionNombre|
+----------------------+--------------------+---------------------+
|                     0|                   0|              Missing|
|                     1|                   2| Customer Credit Note|
|                     2|                   3| Customer Payment ...|
|                     3|                   4|      Customer Refund|
|                     4|                   5|     Supplier Invoice|
|                     5|                   6| Supplier Credit Note|
|                     6|                   7| Supplier Payment ...|
|                     7|                   8|      Supplier Refund|
|                     8|                   9|       Stock Transfer|
|                     9|                  10|          Stock Issue|
|                    10|                  11|        Stock Receipt|
|                    11|                  12| St

#### Carga
Se guarda el resultado en la tabla `TipoTransaccion` de la base de datos destino.

In [15]:
guardar_db(dest_db_connection_string, tiposTransaccion, 'TipoTransaccion', db_user, db_psswd)
print("TipoTransaccion guardado")

TipoTransaccion guardado


In [16]:
sql_check = '''(SELECT * FROM WWImportersTransactional.movimientos LIMIT 5) AS Temp_check'''
check = obtener_dataframe_de_bd(source_db_connection_string, sql_check, db_user, db_psswd)
check.show()
check.printSchema()

+---------------------+----------+-----------------+---------+---------+-----------+---------------+----------------+--------+
|TransaccionProductoID|ProductoID|TipoTransaccionID|ClienteID|InvoiceID|ProveedorID|OrdenDeCompraID|FechaTransaccion|Cantidad|
+---------------------+----------+-----------------+---------+---------+-----------+---------------+----------------+--------+
|                94344|       108|               10|    185.0|  19763.0|       NULL|           NULL|     Jan 20,2014|   -10.0|
|                96548|       162|               11|      0.0|      0.0|        4.0|          228.0|     Jan 28,2014|    10.0|
|                96560|       216|               10|    474.0|  20224.0|       NULL|           NULL|     Jan 28,2014|   -10.0|
|                96568|        22|               11|      0.0|      0.0|        7.0|          193.0|     Jan 28,2014|    10.0|
|                96648|        25|               11|      0.0|      0.0|        7.0|          408.0|     Jan 28

### Dimensión Fecha

Dimensión generada a partir de las fechas distintas presentes en `movimientos` (columna `FechaTransaccion`). Se estandariza el formato según la regla de negocio y se derivan los atributos de calendario.

#### Extracción
Se extraen las fechas distintas de transacción desde `movimientos`.

In [17]:
sql_fechas = '''(SELECT DISTINCT FechaTransaccion FROM WWImportersTransactional.movimientos) AS Temp_fechas'''
fechas_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_fechas, db_user, db_psswd)
fechas_raw.show(5)

+----------------+
|FechaTransaccion|
+----------------+
|     Jan 20,2014|
|     Jan 28,2014|
|     Feb 01,2014|
|     Mar 25,2014|
|     May 01,2014|
+----------------+
only showing top 5 rows



#### Transformación
Se estandariza el formato de fecha (algunas vienen como "Jan 20,2014" y deben pasar a "YYYY-MM-DD") y se derivan Dia, Mes, Annio y el número de semana ISO.

In [18]:
def estandarizar_fecha(valor):
    if valor is None:
        return None
    valor = valor.strip()
    for fmt in ['%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%b %d,%Y', '%b %d, %Y']:
        try:
            return datetime.strptime(valor, fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return None

udf_fecha = udf(estandarizar_fecha, t.StringType())

fechas = fechas_raw.withColumn('FechaEstandar', udf_fecha(col('FechaTransaccion')))
fechas = fechas.withColumn('Fecha', f.to_date(col('FechaEstandar'), 'yyyy-MM-dd'))
fechas = fechas.select('Fecha').distinct().dropna()

fechas = fechas.withColumn('Dia', f.dayofmonth('Fecha'))
fechas = fechas.withColumn('Mes', f.month('Fecha'))
fechas = fechas.withColumn('Annio', f.year('Fecha'))
fechas = fechas.withColumn('Numero_semana_ISO', f.weekofyear('Fecha'))

fechas = fechas.coalesce(1).withColumn('ID_Fecha', f.monotonically_increasing_id() + 1)
fechas = fechas.select('ID_Fecha', 'Fecha', 'Dia', 'Mes', 'Annio', 'Numero_semana_ISO').orderBy('Fecha')

# Registro comodín ID=0
from pyspark.sql.types import StructType, StructField, IntegerType, DateType

schema_fecha_0 = StructType([
    StructField('ID_Fecha', IntegerType(), True),
    StructField('Fecha', DateType(), True),
    StructField('Dia', IntegerType(), True),
    StructField('Mes', IntegerType(), True),
    StructField('Annio', IntegerType(), True),
    StructField('Numero_semana_ISO', IntegerType(), True),
])

fecha_0 = spark.createDataFrame([(0, None, 0, 0, 0, 0)], schema=schema_fecha_0)
fechas = fechas.union(fecha_0)
fechas.show(5)

+--------+----------+---+---+-----+-----------------+
|ID_Fecha|     Fecha|Dia|Mes|Annio|Numero_semana_ISO|
+--------+----------+---+---+-----+-----------------+
|     233|2013-12-31| 31| 12| 2013|                1|
|     721|2014-01-01|  1|  1| 2014|                1|
|     451|2014-01-02|  2|  1| 2014|                1|
|     108|2014-01-03|  3|  1| 2014|                1|
|     745|2014-01-04|  4|  1| 2014|                1|
+--------+----------+---+---+-----+-----------------+
only showing top 5 rows



#### Carga
Se guarda el resultado en la tabla `Fecha` de la base de datos destino.

In [19]:
guardar_db(dest_db_connection_string, fechas, 'Fecha', db_user, db_psswd)
print("Fecha guardado")

Fecha guardado


### Hecho_Movimientos

Tabla de hechos del proceso de negocio. Se eliminan duplicados totales, se estandariza la fecha y se hace left join con las 5 dimensiones, usando el registro comodín (ID=0) cuando no hay coincidencia para no perder filas.

#### Extracción
Se extraen todos los movimientos de inventario desde `movimientos`.

In [20]:
sql_movimientos = '''(SELECT DISTINCT * FROM WWImportersTransactional.movimientos) AS Temp_movimientos'''
movimientos = obtener_dataframe_de_bd(source_db_connection_string, sql_movimientos, db_user, db_psswd)

#### Transformación
Se eliminan duplicados totales, se estandariza `FechaTransaccion` y se hace join con las dimensiones Producto, Cliente, Proveedor, TipoTransaccion y Fecha. `InvoiceID` y `OrdenDeCompraID` se conservan como dimensiones degeneradas. Se mantiene el signo de `Cantidad` (negativo = salida de inventario, según indicó WWI).

In [21]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [22]:
print((movimientos.count(), movimientos.distinct().count()))
movimientos = movimientos.drop_duplicates()

movimientos = movimientos.repartition(8)
movimientos = movimientos.withColumn('FechaEstandar', udf_fecha(col('FechaTransaccion')))
movimientos = movimientos.withColumn('Fecha', f.to_date(col('FechaEstandar'), 'yyyy-MM-dd'))

hecho_movimientos = movimientos.alias('m') \
    .join(productos.alias('pr'), col('m.ProductoID') == col('pr.ID_Producto_T'), 'left') \
    .join(clientes.alias('cl'), col('m.ClienteID') == col('cl.ID_Cliente_T'), 'left') \
    .join(mapa_proveedor.alias('pv'), col('m.ProveedorID') == col('pv.ID_Proveedor_T'), 'left') \
    .join(tiposTransaccion.alias('tt'), col('m.TipoTransaccionID') == col('tt.ID_TipoTransaccion_T'), 'left') \
    .join(fechas.alias('f'), col('m.Fecha') == col('f.Fecha'), 'left') \
    .select(
        col('m.TransaccionProductoID'),
        col('pr.ID_Producto_DWH'),
        col('cl.ID_Cliente_DWH'),
        col('pv.ID_Proveedor_DWH'),
        col('tt.ID_TipoTransaccion_DWH'),
        col('f.ID_Fecha'),
        col('m.InvoiceID'),
        col('m.OrdenDeCompraID'),
        col('m.Cantidad')
    ) \
    .fillna({'ID_Producto_DWH': 0, 'ID_Cliente_DWH': 0, 'ID_Proveedor_DWH': 0,
             'ID_TipoTransaccion_DWH': 0, 'ID_Fecha': 0})

hecho_movimientos.show(5)

(236667, 236667)


26/07/01 17:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 1

+---------------------+---------------+--------------+----------------+----------------------+--------+---------+---------------+--------+
|TransaccionProductoID|ID_Producto_DWH|ID_Cliente_DWH|ID_Proveedor_DWH|ID_TipoTransaccion_DWH|ID_Fecha|InvoiceID|OrdenDeCompraID|Cantidad|
+---------------------+---------------+--------------+----------------+----------------------+--------+---------+---------------+--------+
|               236980|             45|           187|               0|                     9|     151|  49634.0|           NULL|   -24.0|
|               195991|            202|             0|               8|                    10|       0|      0.0|         1880.0|     5.0|
|                 5946|             82|             0|               4|                    10|       0|      0.0|          397.0|    72.0|
|               252962|             44|            61|               0|                     9|       0|  52993.0|           NULL|   -84.0|
|               205109|    

#### Carga
Se guarda el resultado en la tabla `Hecho_Movimientos` de la base de datos destino.

In [23]:
from pyspark.sql.functions import to_date, coalesce as spark_coalesce

movimientos = movimientos.drop_duplicates()

# Estandarizar fecha sin UDF (más rápido): intenta varios formatos con funciones nativas
movimientos = movimientos.withColumn(
    'Fecha',
    spark_coalesce(
        to_date(col('FechaTransaccion'), 'yyyy-MM-dd HH:mm:ss'),
        to_date(col('FechaTransaccion'), 'yyyy-MM-dd'),
        to_date(col('FechaTransaccion'), 'MMM dd,yyyy')
    )
)

hecho_movimientos = movimientos.alias('m') \
    .join(productos.alias('pr'), col('m.ProductoID') == col('pr.ID_Producto_T'), 'left') \
    .join(clientes.alias('cl'), col('m.ClienteID') == col('cl.ID_Cliente_T'), 'left') \
    .join(mapa_proveedor.alias('pv'), col('m.ProveedorID') == col('pv.ID_Proveedor_T'), 'left') \
    .join(tiposTransaccion.alias('tt'), col('m.TipoTransaccionID') == col('tt.ID_TipoTransaccion_T'), 'left') \
    .join(fechas.alias('f'), col('m.Fecha') == col('f.Fecha'), 'left') \
    .select(
        col('m.TransaccionProductoID'),
        col('pr.ID_Producto_DWH'),
        col('cl.ID_Cliente_DWH'),
        col('pv.ID_Proveedor_DWH'),
        col('tt.ID_TipoTransaccion_DWH'),
        col('f.ID_Fecha'),
        col('m.InvoiceID'),
        col('m.OrdenDeCompraID'),
        col('m.Cantidad')
    ) \
    .fillna({'ID_Producto_DWH': 0, 'ID_Cliente_DWH': 0, 'ID_Proveedor_DWH': 0,
             'ID_TipoTransaccion_DWH': 0, 'ID_Fecha': 0})

hecho_movimientos.show(5)

26/07/01 17:43:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 1

+---------------------+---------------+--------------+----------------+----------------------+--------+---------+---------------+--------+
|TransaccionProductoID|ID_Producto_DWH|ID_Cliente_DWH|ID_Proveedor_DWH|ID_TipoTransaccion_DWH|ID_Fecha|InvoiceID|OrdenDeCompraID|Cantidad|
+---------------------+---------------+--------------+----------------+----------------------+--------+---------+---------------+--------+
|               125089|            144|             0|               8|                    10|     512|      0.0|          140.0|    48.0|
|               210818|             27|             0|               8|                    10|     310|      0.0|          654.0|     1.0|
|               164250|            212|             0|               8|                    10|     186|      0.0|         1815.0|   100.0|
|               305228|            196|             0|               4|                    10|     514|      0.0|         1901.0| 40536.0|
|               252194|    

In [24]:
import time
inicio = time.time()
guardar_db(dest_db_connection_string, hecho_movimientos, 'Hecho_Movimientos', db_user, db_psswd)
print(f"Hecho_Movimientos guardado en {time.time() - inicio:.1f} segundos")

26/07/01 17:43:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 17:43:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/01 1

Hecho_Movimientos guardado en 12.2 segundos


In [25]:
for tabla in ['Proveedor', 'TipoTransaccion', 'Fecha', 'Hecho_Movimientos']:
    sql_v = f'''(SELECT COUNT(*) as total FROM DB_202613_c_cantini.{tabla}) AS Temp_v'''
    r = obtener_dataframe_de_bd(dest_db_connection_string, sql_v, db_user, db_psswd)
    print(tabla, '->', r.collect()[0]['total'], 'filas')

Proveedor -> 96 filas


TipoTransaccion -> 78 filas
Fecha -> 3795 filas
Hecho_Movimientos -> 236667 filas
